# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#!pip install moviepy==1.0.3
#!pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_11972\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 3 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (27 articulos)...
  Procesando: Urbanismo retoma la protección de 336 edificios, bienes...
    -> Texto completo (4491 chars)
  Procesando: Los arquitectos apoyan la revisión integral de las norm...
    -> Texto completo (3067 chars)
  Procesando: Zuheros sigue coleccionando títulos por su encanto: aho...
    -> Texto completo (2888 chars)
  Procesando: Moreno adelanta las elecciones en Andalucía al 17 de ma...
    -> Texto completo (7715 chars)
  Procesando: El Consejo de Estado falla contra el registro de jornad...
    -> Texto completo (8008 chars)
  Procesando: Podemos se abstendrá el jueves en la votación del d

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [2]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  Urbanismo retoma la protección de 336 edificios, bienes o hitos urbanos de Córdoba
Origen:  completo
Chars:   4491
Preview: Córdoba
El incendio ocurrido este sábado en la iglesia Madre de Dios, situada en la zona de Campo Madre de Dios, fuera del perímetro del casco histórico de Córdoba trae a primer plano un largo asunto ...

Artículo 2
Fuente:  ABC
Título:  Los arquitectos apoyan la revisión integral de las normas urbanísticas para «adaptarse a la realidad» de Córdoba
Origen:  completo
Chars:   3067
Preview: Córdoba
Urbanismo pretende aprobar este martes una actualización de sus normas urbanísticas que busca revisar y depurar las directrices actuales -con 25 años de vigencia- y que afectará a casi el 40% ...

Artículo 3
Fuente:  ABC
Título:  Zuheros sigue coleccionando títulos por su encanto: ahora, 'Municipio Turístico'
Origen:  completo
Chars:   2888
Preview: Córdoba
El Pleno del Consejo Andaluz del Turismo, presidido este lunes por el consejero de

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [8]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.0-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 27 ARTÍCULOS

-> Agrupando artículos por temas...
   Error agrupando temas: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 861.432289ms. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token

TypeError: cannot unpack non-iterable NoneType object

## Paso 3: Síntesis de voz (T2S)

In [ ]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       4485
  Audio:       audios/audio_2026-03-23_20-24-16.mp3
Audio generado! (1.55 MB)
  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

In [ ]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 11449 chars
WEBVTT

00:00:00.000 --> 00:00:01.760
Conflicto y negociaciones entre

00:00:01.760 --> 00:00:02.980
Estados Unidos y Irán.

00:00:04.019 --> 00:00:05.099
El presidente de Estados

00:00:05.099 --> 00:00:07.400
Unidos, Donald Trump, anunció

00:00:07.400 --> 00:00:08.320
la orden de posponer

00:00:


## Paso 4: Generación del video resumen 

In [ ]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')

ruta_video = generar_video(ruta_audio, ruta_subtitulos, resumenes, PEXELS_API_KEY)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    290.5s
   Temas:             6
   Duración por tema: 48.4s

2. Buscando imágenes en Pexels...
  Tema: Conflicto y Negociaciones entre EEUU e Irán...
    Keywords: 'conflicto negociaciones entre eeuu'
    Imagen guardada: imagenes/img_8842911119247063826.jpg
  Tema: Crímenes y Sucesos en España...
    Keywords: 'crímenes sucesos españa'
    Imagen guardada: imagenes/img_8572434273152678577.jpg
  Tema: Noticias Regionales y Locales de España...
    Keywords: 'noticias regionales locales españa'
    Imagen guardada: imagenes/img_7089684556216950599.jpg
  Tema: Crisis Energética Global...
    Keywords: 'crisis energética global'
    Imagen guardada: imagenes/img_251657473663213057.jpg
  Tema: Geopolítica: Preparación Europea...
    Keywords: 'geopolítica preparación europea'
    Imagen guardada: imagenes/img_7432720517247729759.jpg
  Tema: Deportes: Atletismo Español...
    Keywords: 'deportes atletismo español'
    Imagen guarda